# ShelfSense Sprint 4: YOLOv8n on SKU-110K + pre-labels for our frames

Sprint 3 showed the VLM cannot count products. This notebook trains the **detector** that will do the counting:

1. Train **YOLOv8n** (nano, same as `configs/config.yaml`) on a subset of **SKU-110K**
   (dense supermarket shelf photos, one class: `product`).
2. Report **mAP** on held-out SKU-110K validation images.
3. Run the trained model on **our** annotation frames (from `scripts/select_annotation_frames.py`) and save its
   boxes as **pre-labels**. Annotators then only correct boxes (delete wrong ones, add missed ones,
   draw `empty_gap`) instead of drawing ~50-100 boxes per frame from scratch.

### How to run on Kaggle
1. On your laptop (PowerShell, project root, venv active):
   ```
   python scripts/select_annotation_frames.py --per-video 2
   tar -a -c -f data\annotation_batch\annotate_train.zip -C data\annotation_batch train
   tar -a -c -f data\annotation_batch\annotate_test.zip -C data\annotation_batch test
   ```
   (use `tar`, not `Compress-Archive`: its zips break on Kaggle's Linux)
2. Kaggle → **Datasets → New Dataset** → upload both zips, name it `shelfsense-annotation-batch`
3. Kaggle → **Code → New Notebook** → File → Import Notebook → this `.ipynb`
4. Right panel: **Add Input** → `shelfsense-annotation-batch`.
   **Settings**: Accelerator = **GPU T4** (or P100), **Internet = On**
5. **Save Version → Save & Run All (Commit)**. It runs in the background, so you can close the browser.
   Expect about 1-1.5 h: ~15 min to download and prepare SKU-110K, ~45-60 min training, a few min pre-labelling.
6. From the version's **Output** tab download:
   - `prelabels_roboflow.zip`: our frames + pre-labels (YOLO format, classes `product`, `empty_gap`)
   - `yolo_sku110k_summary.json`, `prelabel_counts.csv`, `prelabel_examples.jpg`: numbers and a figure for the report
   - `runs/sku110k_v8n/results.png` and `results.csv`: training curves
   - `sku110k_yolov8n_best.pt`: the weights. Put them on **Google Drive**, never in git.

### Then, in Roboflow (free)
New project → **Object Detection** → upload `prelabels_roboflow.zip` (it is read as a YOLO dataset with the
`train` and `test` splits). For each frame: delete wrong boxes, add missed products, draw `empty_gap` boxes.
**Test frames are ground truth**: two team members must check each one, because pre-labels pull the
annotator towards what the model already sees (which would make the model look better than it is).

### What the numbers mean
mAP here is measured on SKU-110K photos (similar shelves, different country and camera). The real test is
on **our** test frames once they are corrected; that is the next notebook (fine-tune on our train frames,
evaluate on our test frames).

In [ ]:
import json, random, shutil, subprocess, time
from pathlib import Path

SEED = 0
N_TRAIN = 2000            # SKU-110K training images used (of ~8,200): enough for one class, fits the time budget
N_VAL = 500               # SKU-110K validation images used for mAP
MAX_SIDE = 1280           # SKU-110K photos are up to ~4000 px; shrink them once so training is not CPU-bound
IMGSZ = 1024              # our frames are 576x1024, so 1024 keeps them at full detail
EPOCHS = 30
BATCH = 16
MAX_DET = 1000            # default is 300; a packed shelf can hold more products than that
PRELABEL_CONF = 0.25      # boxes below this confidence are not written as pre-labels

SKU_URL = "http://trax-geometry.s3.amazonaws.com/cvpr_challenge/SKU110K_fixed.tar.gz"
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
TMP = Path("/tmp/shelfsense")          # large data goes here: /kaggle/working is limited to 20 GB
SKU_RAW = TMP / "SKU110K_fixed"
SKU_YOLO = TMP / "sku110k_yolo"
TMP.mkdir(parents=True, exist_ok=True)
print(f"free disk for /tmp: {shutil.disk_usage(TMP).free / 1e9:.0f} GB (SKU-110K needs about 14 GB)")

In [ ]:
!pip install -q ultralytics

In [ ]:
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings")

# Our frames: Kaggle unzips annotate_train.zip / annotate_test.zip into folders named train/ and test/
ours = {"train": {}, "test": {}}
for p in sorted(INPUT.rglob("*.jpg")):
    if "__frame_" in p.name and p.parent.name in ours:
        ours[p.parent.name][p.name] = p      # keyed by name, so a frame is never used twice
ours = {split: list(frames.values()) for split, frames in ours.items()}
print({split: len(frames) for split, frames in ours.items()}, "of our frames found")
assert ours["train"], "No frames found: did you add the shelfsense-annotation-batch dataset as input?"

## 1. Get SKU-110K
Downloaded (13.6 GB) and unpacked in one stream, so the `.tar.gz` is never stored on disk.
If you attach a Kaggle copy of SKU-110K that has the original `annotations/` CSVs, it is used instead.

In [ ]:
attached = sorted(INPUT.rglob("annotations_train.csv"))
if attached:
    SKU_RAW = attached[0].parent.parent
elif not (SKU_RAW / "annotations" / "annotations_train.csv").exists():
    t = time.time()
    subprocess.run(f"set -o pipefail; curl -sSL {SKU_URL} | tar -xz -C {TMP}",
                   shell=True, check=True, executable="/bin/bash")
    print(f"downloaded and unpacked in {(time.time() - t) / 60:.1f} min")
print(SKU_RAW, "-", len(list((SKU_RAW / "images").glob("*.jpg"))), "images")

## 2. Convert a random subset to YOLO format
SKU-110K's CSVs have no header: `image, x1, y1, x2, y2, class, image_width, image_height` (pixel corners).
YOLO wants one `.txt` per image with `class x_center y_center width height`, all divided by the image size.
Every box is class 0 = `product` (no brands, as planned). Images whose size does not match the CSV
(rotated by EXIF) or that are corrupt are skipped rather than guessed.

In [ ]:
COLS = ["image", "x1", "y1", "x2", "y2", "class", "image_width", "image_height"]


def convert(csv_name, split, n):
    df = pd.read_csv(SKU_RAW / "annotations" / csv_name, names=COLS)
    names = sorted(df["image"].unique())
    random.Random(SEED).shuffle(names)
    groups = df.groupby("image")
    img_dir, lbl_dir = SKU_YOLO / "images" / split, SKU_YOLO / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    done, skipped, n_boxes = 0, 0, 0
    for name in names:
        if done == n:
            break
        im = cv2.imread(str(SKU_RAW / "images" / name))
        g = groups.get_group(name)
        w, h = int(g["image_width"].iloc[0]), int(g["image_height"].iloc[0])
        if im is None or (im.shape[1], im.shape[0]) != (w, h):
            skipped += 1
            continue
        x1, x2 = g["x1"].clip(0, w), g["x2"].clip(0, w)
        y1, y2 = g["y1"].clip(0, h), g["y2"].clip(0, h)
        ok = (x2 > x1) & (y2 > y1)
        lines = [f"0 {(a + c) / 2 / w:.5f} {(b + d) / 2 / h:.5f} {(c - a) / w:.5f} {(d - b) / h:.5f}"
                 for a, b, c, d in zip(x1[ok], y1[ok], x2[ok], y2[ok])]
        scale = MAX_SIDE / max(w, h)
        if scale < 1:   # labels are relative to the image size, so shrinking does not change them
            im = cv2.resize(im, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(img_dir / name), im, [cv2.IMWRITE_JPEG_QUALITY, 90])
        (lbl_dir / name).with_suffix(".txt").write_text("\n".join(lines) + "\n")
        done += 1
        n_boxes += len(lines)
    print(f"{split}: {done} images, {n_boxes} boxes ({n_boxes / max(done, 1):.0f} per image), {skipped} skipped")
    return {"images": done, "boxes": n_boxes, "skipped": skipped}


t = time.time()
if SKU_YOLO.exists():
    shutil.rmtree(SKU_YOLO)
sku_subset = {"train": convert("annotations_train.csv", "train", N_TRAIN),
              "val": convert("annotations_val.csv", "val", N_VAL)}
(SKU_YOLO / "data.yaml").write_text(f"path: {SKU_YOLO}\ntrain: images/train\nval: images/val\nnames:\n  0: product\n")
print(f"converted in {(time.time() - t) / 60:.1f} min")

## 3. Train YOLOv8n
Starts from the COCO-pretrained nano model. `patience=10` stops early if validation mAP has not improved
for 10 epochs. Curves and the per-epoch numbers are saved in `runs/sku110k_v8n/`.

In [ ]:
t = time.time()
model = YOLO("yolov8n.pt")
model.train(data=str(SKU_YOLO / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, max_det=MAX_DET,
            device=0, workers=4, seed=SEED, patience=10,
            project=str(WORK / "runs"), name="sku110k_v8n", exist_ok=True)
train_minutes = round((time.time() - t) / 60, 1)
print(f"training took {train_minutes} min")

In [ ]:
best = WORK / "runs" / "sku110k_v8n" / "weights" / "best.pt"
shutil.copy(best, WORK / "sku110k_yolov8n_best.pt")
model = YOLO(best)
m = model.val(data=str(SKU_YOLO / "data.yaml"), imgsz=IMGSZ, batch=BATCH, max_det=MAX_DET, device=0, plots=False)
sku_val = {"mAP50": round(float(m.box.map50), 3), "mAP50_95": round(float(m.box.map), 3),
           "precision": round(float(m.box.mp), 3), "recall": round(float(m.box.mr), 3)}
print("SKU-110K validation subset:", sku_val)

## 4. Pre-label our frames
Output layout (Roboflow's YOLO import format): `train/images`, `train/labels`, `test/images`, `test/labels`,
`data.yaml`. `data.yaml` already lists `empty_gap` as class 1 so annotators can draw it; the model only
predicts class 0 `product`.

In [ ]:
OUT = WORK / "prelabels"
if OUT.exists():
    shutil.rmtree(OUT)
per_frame = []
for split, frames in ours.items():
    (OUT / split / "images").mkdir(parents=True)
    (OUT / split / "labels").mkdir(parents=True)
    for p in frames:
        r = model.predict(str(p), imgsz=IMGSZ, conf=PRELABEL_CONF, max_det=MAX_DET, device=0, verbose=False)[0]
        shutil.copy(p, OUT / split / "images" / p.name)
        lines = [f"0 {x:.5f} {y:.5f} {w:.5f} {h:.5f}" for x, y, w, h in r.boxes.xywhn.tolist()]
        (OUT / split / "labels" / f"{p.stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        confs = r.boxes.conf.tolist()
        per_frame.append({"split": split, "image": p.name, "boxes": len(lines),
                          "mean_conf": round(sum(confs) / len(confs), 3) if confs else ""})
(OUT / "data.yaml").write_text("train: train/images\nval: test/images\nnc: 2\nnames: ['product', 'empty_gap']\n")
shutil.make_archive(str(WORK / "prelabels_roboflow"), "zip", OUT)
shutil.rmtree(OUT)   # the zip is all that is needed

pd.DataFrame(per_frame).to_csv(WORK / "prelabel_counts.csv", index=False)
counts = pd.DataFrame(per_frame).groupby("split")["boxes"].describe()[["count", "mean", "min", "50%", "max"]]
print(counts.round(1))

In [ ]:
# Figure for the report: 12 test frames spread over the test videos, with predicted boxes
picks = ours["test"][:: max(1, len(ours["test"]) // 12)][:12]
tiles = []
for p in picks:
    r = model.predict(str(p), imgsz=IMGSZ, conf=PRELABEL_CONF, max_det=MAX_DET, device=0, verbose=False)[0]
    im = cv2.resize(r.plot(labels=False, conf=False, line_width=1), (288, 512))
    cv2.putText(im, f"{p.name.split('__')[0][-6:]}: {len(r.boxes)} boxes", (6, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    tiles.append(im)
while len(tiles) % 6:
    tiles.append(np.full_like(tiles[0], 255))
grid = np.vstack([np.hstack(tiles[i:i + 6]) for i in range(0, len(tiles), 6)])
cv2.imwrite(str(WORK / "prelabel_examples.jpg"), grid, [cv2.IMWRITE_JPEG_QUALITY, 85])

from IPython.display import Image, display
display(Image(str(WORK / "prelabel_examples.jpg")))

In [ ]:
summary = {
    "model": "yolov8n.pt (COCO) fine-tuned on SKU-110K subset",
    "settings": {"n_train": N_TRAIN, "n_val": N_VAL, "max_side": MAX_SIDE, "imgsz": IMGSZ, "epochs": EPOCHS,
                 "batch": BATCH, "max_det": MAX_DET, "prelabel_conf": PRELABEL_CONF, "seed": SEED},
    "sku_subset": sku_subset,
    "train_minutes": train_minutes,
    "sku_val": sku_val,
    "prelabels": {split: {"frames": int(row["count"]), "mean_boxes": round(float(row["mean"]), 1),
                          "min_boxes": int(row["min"]), "max_boxes": int(row["max"])}
                  for split, row in counts.iterrows()},
    "gpu": torch.cuda.get_device_name(0),
}
(WORK / "yolo_sku110k_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))